In [0]:
from pyspark import pipelines as dp
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import col, concat_ws, current_timestamp, lit, to_date

@dp.table(
    name="fineguard.gold.high_value_transaction_alert",
    comment="Alert details where transactions are performed with a value higher than the configured transaction limit"
)
def high_value_transaction_alert() -> DataFrame:
    transactions_df = spark.readStream.table("fineguard.silver.transactions")
    customer_df = spark.read.table("fineguard.silver.customers")

    joined_df = (
        transactions_df.join(
            customer_df,
            transactions_df["customer_id"] == customer_df["customer_id"],
            "left"
        )
        .filter(transactions_df["amount"] > customer_df["transaction_limit"])
        .select(
            concat_ws("-", lit("ALERT"), transactions_df["transaction_id"]).alias("alert_id"),
            lit("HIGH_VALUE_TRANSACTION").alias("alert_type"),
            current_timestamp().alias("alert_timestamp"),
            transactions_df["transaction_id"],
            transactions_df["customer_id"],
            customer_df["email"].alias("customer_email_id"),
            concat_ws(" ", customer_df["first_name"], customer_df["last_name"]).alias("customer_name"),
            to_date(transactions_df["transaction_timestamp"]).alias("transaction_date"),
            transactions_df["amount"].alias("transaction_amount"),
            customer_df["transaction_limit"].cast("double").alias("transaction_limit"),
            transactions_df["currency"].alias("transaction_currency"),
            transactions_df["merchant_name"],
            transactions_df["merchant_category"],
            transactions_df["transaction_type"],
            transactions_df["payment_channel"],
            transactions_df["country"].alias("transaction_country"),
            transactions_df["city"].alias("transaction_city"),
            transactions_df["is_international"],
            transactions_df["transaction_timestamp"],
            transactions_df["status"]
        )
    )

    return joined_df
